# Volume vs surface

In [15]:
from collections.abc import Mapping
from dataclasses import fields, is_dataclass
from hashlib import sha256
import json
from pathlib import Path
import sys
from tempfile import NamedTemporaryFile
from time import perf_counter

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import trimesh

root = next(candidate for parent in (Path.cwd(), *Path.cwd().parents)
            for candidate in (parent, parent / "mesh-to-housing")
            if (candidate / "src/mesh_to_housing").is_dir())
sys.path.insert(0, str(root / "src"))

from mesh_to_housing import (
    CombinedLossConfig, Construction, Cuboid, ExcessSurfaceConfig, FittingTask, Greedy, LossTerm,
    MeshLoader, OutsideVolumeConfig, Placement, RandomPool, ResidualProposals, TransformConstraints,
    UncoveredSurfaceConfig, UncoveredVolumeConfig, VoxelGridConfig, fit, resize_mesh,
)

pio.renderers.default = "plotly_mimetype+notebook"


def mesh_fingerprint(mesh):
    digest = sha256()
    for array, dtype in ((mesh.vertices, "<f8"), (mesh.faces, "<i8")):
        values = np.ascontiguousarray(array, dtype=dtype)
        digest.update(str(values.shape).encode())
        digest.update(values.tobytes())
    return digest.hexdigest()


def describe(value):
    if is_dataclass(value):
        return {"type": f"{type(value).__module__}.{type(value).__qualname__}",
                **{field.name: describe(getattr(value, field.name)) for field in fields(value)}}
    if isinstance(value, Mapping):
        return {key: describe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [describe(item) for item in value]
    if isinstance(value, trimesh.Trimesh):
        return {"mesh": mesh_fingerprint(value)}
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def load_or_compute(kind, spec, compute):
    key = sha256(json.dumps(spec, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()
    directory = cache_dir / kind
    path = directory / f"{key}.json"
    if path.exists():
        return key, json.loads(path.read_text())["result"], True
    result = compute()
    directory.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", dir=directory, suffix=".tmp", delete=False) as stream:
        temporary = Path(stream.name)
        try:
            json.dump({"spec": spec, "result": result}, stream, allow_nan=False)
        except BaseException:
            temporary.unlink(missing_ok=True)
            raise
    temporary.replace(path)
    return key, result, False


def cached_fit(algorithm, objective, *, version):
    spec = {"task": task_spec, "algorithm": describe(algorithm), "version": version,
            "objective": describe(objective), "seed": seed}

    def compute():
        started = perf_counter()
        construction = fit(task, algorithm=algorithm, loss=objective.build(), seed=seed)
        seconds = perf_counter() - started
        placements = [{"item": placed.item.key, "center": placed.center.tolist(),
                       "rotation": placed.rotation.tolist(), "scale": placed.scale} for placed in construction]
        return {"placements": placements, "seconds": seconds}

    key, saved, loaded = load_or_compute("fits", spec, compute)
    catalog = {item.key: item for item in task.catalog}
    construction = Construction(tuple(Placement(catalog[placed["item"]], placed["center"],
                                               placed["rotation"], placed["scale"])
                                      for placed in saved["placements"]))
    task.validate(construction)
    print(f"{len(construction)} items · {saved['seconds']:.1f} s · {'cached' if loaded else 'computed'}")
    return {"key": key, "construction": construction, "seconds": saved["seconds"]}

In [16]:
model_path = root / "mesh-assets/dinosaur.dae"
target_height = 5.0
item_budget = 100
seed = 42
catalog = (Cuboid("floor_tile", dimensions=(2.0, 2.0, 0.43322)),)
constraints = TransformConstraints(scale_bounds=(0.2, 2.0))

fit_grid = VoxelGridConfig(resolution=64, gap_closing=0.0)
judge_grid = VoxelGridConfig(resolution=96, gap_closing=0.0)
surface_tolerance = None
pool_settings = dict(centers=256, orientations=8, scales=5)
residual_settings = dict(spots=32, spins=2, scales=5, orientation="surface")
renew_every = 10
volume_weights = dict(uncovered=1.0, outside=1.0)
surface_weights = dict(uncovered=1.0, excess=1.0)
algorithm_versions = dict(volume=1, surface=1)
report_version = 1
cache_dir = root.parent / "fitting-cache" / "surface-baseline"

mesh = resize_mesh(MeshLoader.load(model_path), target_height, axis=2)
task = FittingTask(mesh, catalog, item_budget, constraints)
pool = RandomPool(grid=fit_grid, **pool_settings)
volume_algorithm = Greedy(pool=pool)
surface_algorithm = Greedy(
    pool=pool,
    renewed=ResidualProposals(grid=fit_grid, surface_tolerance=surface_tolerance,
                              skin_only=True, **residual_settings),
    renew_every=renew_every,
)


def volume_objective(grid):
    return CombinedLossConfig(terms={
        "uncovered": LossTerm(UncoveredVolumeConfig(grid=grid), weight=volume_weights["uncovered"]),
        "outside": LossTerm(OutsideVolumeConfig(grid=grid), weight=volume_weights["outside"]),
    })


def surface_objective(grid):
    return CombinedLossConfig(terms={
        "uncovered": LossTerm(UncoveredSurfaceConfig(grid=grid, surface_tolerance=surface_tolerance,
                                                    skin_only=True), weight=surface_weights["uncovered"]),
        "excess": LossTerm(ExcessSurfaceConfig(grid=grid, surface_tolerance=surface_tolerance),
                           weight=surface_weights["excess"]),
    })


task_spec = {
    "target": {"mesh": mesh_fingerprint(task.target), "height": target_height, "axis": 2},
    "catalog": [{"item": describe(item), "mesh": mesh_fingerprint(item.mesh)} for item in task.catalog],
    "budget": task.max_items,
    "constraints": describe(task.constraints),
}

In [ ]:
volume = cached_fit(volume_algorithm, volume_objective(fit_grid), version=algorithm_versions["volume"])

In [ ]:
surface = cached_fit(surface_algorithm, surface_objective(fit_grid), version=algorithm_versions["surface"])

In [ ]:
results = {"Volume": volume, "Surface": surface}
panels = [("Target", task.target, "#8fa6bc")]
for name, result in results.items():
    construction = result["construction"]
    panels.append((f"{name} · {len(construction)} items", construction.to_mesh(), "#b47752"))

vertices = np.concatenate([model.vertices for _, model, _ in panels])
low, high = vertices.min(axis=0), vertices.max(axis=0)
padding = max(float((high - low).max()) * 0.05, 1e-6)
axes = {f"{axis}axis": dict(visible=False, range=[low[index] - padding, high[index] + padding])
        for index, axis in enumerate("xyz")}
fig = make_subplots(rows=1, cols=3, specs=[[{"type": "scene"}] * 3],
                    subplot_titles=[title for title, _, _ in panels], horizontal_spacing=0.02)
for column, (_, model, color) in enumerate(panels, start=1):
    x, y, z = model.vertices.T
    i, j, k = model.faces.T
    fig.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color=color, flatshading=True,
                           showscale=False, hoverinfo="skip"), row=1, col=column)
fig.update_scenes(**axes, aspectmode="data",
                  camera=dict(up=dict(x=0, y=0, z=1), eye=dict(x=1.5, y=-1.5, z=0.8)))
fig.update_layout(template="plotly_white", height=480, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
import pandas as pd

judge_losses = {
    "Uncovered volume": UncoveredVolumeConfig(grid=judge_grid),
    "Outside volume": OutsideVolumeConfig(grid=judge_grid),
    "Uncovered surface · solid": UncoveredSurfaceConfig(
        grid=judge_grid, surface_tolerance=surface_tolerance,
    ),
    "Uncovered surface · skin": UncoveredSurfaceConfig(
        grid=judge_grid, surface_tolerance=surface_tolerance, skin_only=True,
    ),
    "Excess surface": ExcessSurfaceConfig(
        grid=judge_grid, surface_tolerance=surface_tolerance,
    ),
    "Volume objective": volume_objective(judge_grid),
    "Surface objective": surface_objective(judge_grid),
}

rows = []
for name, result in results.items():
    spec = {
        "fit": result["key"],
        "losses": describe(judge_losses),
        "version": report_version,
    }

    def evaluate():
        return {
            label: float(config.build()(task.target, result["construction"]))
            for label, config in judge_losses.items()
        }

    _, losses, _ = load_or_compute("reports", spec, evaluate)
    rows.append({
        "Fit": name,
        "Items": len(result["construction"]),
        "Fit (s)": result["seconds"],
        **losses,
    })

pd.DataFrame(rows).set_index("Fit").round(4)